# Appendix

The Appendix contains all code relevant for the methodology, strictly from .py files. All evaluations, treated as part of the Appendix are accessible in the following github repository: https://github.com/kovachna/nonlinear_metrics_thesis

In [ ]:
import numpy as np
import xarray as xr
from pathlib import Path
from typing import Union
import matplotlib.pyplot as plt
from scipy.signal import detrend, butter, filtfilt
from preprocessing import preprocess_signal, _butter_filter

def approximate_entropy(U: np.ndarray, m: int = 2, r: float = None) -> float:
    """
    Calculates Approximate Entropy (ApEn) with m=2, r=0.2*SD default.
    On displacement signal.
    """
    U = np.asarray(U)
    N = len(U)
    if r is None:
        r = 0.2 * np.std(U)
    
    def phi(mv):
        X = np.array([U[i:i+mv] for i in range(N-mv+1)])
        C = []
        for xi in X:
            # Count patterns within tolerance r
            matches = np.sum(np.max(np.abs(X - xi), axis=1) <= r)
            C.append(matches / (N - mv + 1))
        # Avoid log(0) by adding small epsilon
        C = np.array(C)
        C[C == 0] = np.finfo(float).eps
        return np.mean(np.log(C))
    
    return phi(m) - phi(m+1)

def sample_entropy(U: np.ndarray, m: int = 2, r: float = None) -> float:
    """
    Calcautes Sample Entropy (SampEn) with m=2, r=0.2*SD default.
    On displacement signal.
    """
    U = np.asarray(U)
    N = len(U)
    if r is None:
        r = 0.2 * np.std(U)
    
    # Templates of length m
    X = np.array([U[i:i+m] for i in range(N-m+1)])
    # Templates of length m+1
    X1 = np.array([U[i:i+m+1] for i in range(N-m)])
    
    def count_matches(Xarr):
        cnt = 0
        for i in range(len(Xarr)):
            for j in range(i+1, len(Xarr)):
                if np.max(np.abs(Xarr[i] - Xarr[j])) <= r:
                    cnt += 1
        return cnt
    
    B = count_matches(X)
    A = count_matches(X1)
    
    if B == 0 or A == 0:
        return np.inf
    else:
        return -np.log(A / B)

def coarse_grain(ts: np.ndarray, scale: int) -> np.ndarray:
    """Non-overlapping average for MSE coarse-graining."""
    N = len(ts)
    num = N // scale
    return np.mean(ts[:num*scale].reshape(num, scale), axis=1)

def multiscale_entropy(ts: np.ndarray,
                      m: int = 2,
                      r: float = None,
                      max_scale: int = 20) -> tuple[np.ndarray, np.ndarray]:
    """
    Multiscale Entropy (MSE): SampEn on each coarse-grained series.
    """
    if r is None:
        r = 0.2 * np.std(ts)
    
    scales = np.arange(1, max_scale + 1)
    mse = []
    
    for scale in scales:
        # Coarse-grain the time series
        coarse_ts = coarse_grain(ts, scale)
        # Compute sample entropy on coarse-grained series
        se = sample_entropy(coarse_ts, m=m, r=r)
        mse.append(se)
    
    return scales, np.array(mse)

def compute_entropy_on_displacement(ts_disp, fs=100.0, r_factor=0.2):
    """
    Compute ApEn and SampEn directly on displacement signal.
    """
    # Normalize displacement signal
    ts_norm = (ts_disp - np.mean(ts_disp)) / np.std(ts_disp)
    
    # Set tolerance
    r = r_factor * np.std(ts_norm) 
    
    # Compute entropies
    apen = approximate_entropy(ts_norm, m=2, r=r)
    sampen = sample_entropy(ts_norm, m=2, r=r)
    
    return apen, sampen

def compute_entropy_on_acceleration(ts_disp, fs=100.0, r_factor=0.2):
    """
    Compute ApEn and SampEn on acceleration derived from displacement.
    """
    # Compute velocity and acceleration
    vel = np.gradient(ts_disp, 1/fs)
    acc = np.gradient(vel, 1/fs)
    
    # Band-pass filter acceleration
    nyq = fs / 2
    b, a = butter(4, [0.5/nyq, 10/nyq], btype='bandpass')
    acc_f = filtfilt(b, a, acc)
    
    # Z-score normalization
    acc_z = (acc_f - np.mean(acc_f)) / np.std(acc_f)
    
    # Compute entropies with tolerance
    r = r_factor * np.std(acc_z)  # This will be r_factor since acc_z has std=1
    apen = approximate_entropy(acc_z, m=2, r=r)
    sampen = sample_entropy(acc_z, m=2, r=r)
    
    return apen, sampen

def compute_MSE_on_acceleration(ts_disp, r_factor=0.2, fs=100.0, max_scale=20):
    """
    Compute Multiscale Entropy on acceleration derived from displacement.
    """
    # Compute velocity and acceleration
    vel = np.gradient(ts_disp, 1/fs)
    acc = np.gradient(vel, 1/fs)
    
    # Band-pass filter acceleration
    nyq = fs / 2
    b, a = butter(4, [0.5/nyq, 10/nyq], btype='bandpass')
    acc_f = filtfilt(b, a, acc)
    
    # Z-score normalization
    acc_z = (acc_f - np.mean(acc_f)) / np.std(acc_f)
    
    # Compute MSE on acc
    r = r_factor * np.std(acc_z) 
    scales, mse = multiscale_entropy(acc_z, m=2, r=r, max_scale=max_scale)
    
    return scales, mse

In [ ]:

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from numpy.linalg import pinv, eig
from scipy import signal
from sklearn.utils import resample


def apply_lowpass_filter(data: np.ndarray, 
                        fs: float, 
                        cutoff: float = 8.0,
                        order: int = 4) -> np.ndarray:
    """Applies zero-lag Butterworth low-pass filter to data."""
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = signal.butter(order, normal_cutoff, btype='low', analog=False)
    
    filtered_data = np.zeros_like(data)
    for ax in range(data.shape[1]):
        filtered_data[:, ax] = signal.filtfilt(b, a, data[:, ax])
    
    return filtered_data


def find_fixed_point(S: np.ndarray, max_iter: int = 20, tol: float = 1e-6) -> np.ndarray:
    """
    Find fixed point of Poincaré map where x_{k+1} = x_k.

    """
    X = S[:-1, :]  # States at stride k
    Y = S[1:, :]   # States at stride k+1
    
    # Initial guess: mean state
    x_star = S.mean(axis=0)
    
    for i in range(max_iter):
        # Find closest point in X to current guess
        distances = np.sum((X - x_star)**2, axis=1)
        idx = np.argmin(distances)
        
        # The fixed point should satisfy: F(x_star) = x_star
        # So Y[idx] should equal X[idx] at the fixed point
        x_star_new = 0.5 * (X[idx, :] + Y[idx, :])
        
        # Check convergence
        if np.linalg.norm(x_star_new - x_star) < tol:
            
            break
        x_star = x_star_new
    else:
        None
    
    return x_star


def compute_floquet_from_events(com_da: xr.DataArray,
                                events_df: pd.DataFrame,
                                channel: str = "COM",
                                axes: list = ["x", "y", "z"],
                                normalization_points: int = 101,
                                rcond: float = 1e-6,
                                phase_dependent: bool = False,
                                use_same_side: bool = True,
                                force_side: str = None,
                                trim_cycles: int = 5,
                                bootstrap_n: int = 100,
                                bootstrap_ci: float = 95.0,
                                lowpass_cutoff: float = 8.0,
                                return_all_eigenvalues: bool = False,
                                use_fixed_point: bool = True,
                                detrend_y: bool = True) -> dict:
    """
    Compute Floquet multipliers on events from a DataArray of center of mass (COM) data.
    Parametrs are dynamic.
    """
    # Detrend y-axis if requested - extracts the speed on treadmill
    com_da_processed = com_da.copy()
    if detrend_y and 'y' in axes:
        y_data = com_da.sel(channel=channel, axis='y').values
        t_secs = (com_da.time.values - com_da.time.values[0])  # array of seconds
        coeffs = np.polyfit(t_secs, y_data, 1)
        trend  = np.polyval(coeffs, t_secs)
        y_detrended = y_data - trend    
        time_points = np.arange(len(y_data))
        
        # Fit linear trend
        coeffs = np.polyfit(time_points, y_data, 1)
        trend = np.polyval(coeffs, time_points)
        y_detrended = y_data - trend
        
        com_da_processed.loc[dict(channel=channel, axis='y')] = y_detrended
        print(f"Removed drift from y-axis: {trend[-1] - trend[0]:.1f} units total")
    
    # Extract all Foot Strike times and side (context)
    df_fs = events_df[events_df['label'] == 'Foot Strike'].copy()
    df_fs = df_fs[['time','context']].sort_values('time')
    strike_times = df_fs['time'].values
    strike_context = df_fs['context'].values
    
    # Determine which strikes to use based on same-side setting
    if use_same_side:
        left_indices = np.where(strike_context == 'Left')[0]
        right_indices = np.where(strike_context == 'Right')[0]
        
        if force_side is not None:
            # Use the forced side
            side_used = force_side
            use_indices = left_indices if force_side == 'Left' else right_indices
        else:
            # Auto-select the side with more strikes, if no side defined 
            if len(left_indices) >= len(right_indices):
                use_indices = left_indices
                side_used = 'Left'
            else:
                use_indices = right_indices
                side_used = 'Right'
        
        # Create stride segments from same-side strikes
        stride_starts = use_indices[:-1]
        stride_ends = use_indices[1:]
        print(f"Using {side_used}-to-{side_used} strides: {len(stride_starts)} strides")
    else:
        # Use all consecutive strikes
        stride_starts = np.arange(len(strike_times) - 1)
        stride_ends = stride_starts + 1
        side_used = 'both'
        print(f"Using all consecutive strikes: {len(stride_starts)} strides")
    
    # Build raw segments and store stride durations
    raw_segs = []
    raw_ts = []
    raw_contexts = []
    stride_durations = []
    
    for i, (start_idx, end_idx) in enumerate(zip(stride_starts, stride_ends)):
        t0 = strike_times[start_idx]
        t1 = strike_times[end_idx]
        
        seg = (com_da_processed
               .sel(channel=channel, axis=axes)
               .sel(time=slice(t0, t1))
               .transpose('time', 'axis'))
        t = seg.time.values
        
        if t.size < 10:  # if segment very short, skip
            continue
            
        raw_segs.append(seg.values)
        raw_ts.append(t)
        raw_contexts.append(strike_context[start_idx])
        stride_durations.append(t1 - t0)  # Store stride duration
    
    n_cycles = len(raw_segs)
    n_axes = len(axes)
    state_dim = 2 * n_axes

    
    if n_cycles < state_dim + 1 + 2*trim_cycles:
        raise ValueError(f"Need ≥{state_dim + 1 + 2*trim_cycles} valid strides, got {n_cycles}")
    
    # Calculate mean stride duration BEFORE trimming 
    mean_duration = np.mean(stride_durations)
    print(f"Mean stride duration: {mean_duration:.3f} seconds")
    
    # Apply low-pass filter to each segment before normalization
    filtered_segs = []
    for i, (seg_vals, t) in enumerate(zip(raw_segs, raw_ts)):
        fs = 1.0 / np.mean(np.diff(t))
        filtered_seg = apply_lowpass_filter(seg_vals, fs, lowpass_cutoff)
        filtered_segs.append(filtered_seg)
    
    # Time-normalize each cycle while preserving actual time scale
    M = normalization_points
    norm_t = np.linspace(0, 1, M)
    com_norm = np.zeros((n_cycles, M, n_axes), dtype=float)
    
    for i, (seg_vals, t) in enumerate(zip(filtered_segs, raw_ts)):
        t_norm = (t - t[0]) / (t[-1] - t[0])
        for ax in range(n_axes):
            com_norm[i, :, ax] = np.interp(norm_t, t_norm, seg_vals[:, ax])
    
    # Calculated velocity
    dt_actual = mean_duration / (M - 1)
    
    # Trim first and last cycles AFTER calculating mean duration
    if trim_cycles > 0:
        com_norm = com_norm[trim_cycles:-trim_cycles]
        raw_contexts = raw_contexts[trim_cycles:-trim_cycles]
        stride_durations = stride_durations[trim_cycles:-trim_cycles]
        n_cycles_trimmed = n_cycles - 2*trim_cycles
        print(f"Trimmed {trim_cycles} cycles from start/end, using {n_cycles_trimmed} cycles")
    else:
        n_cycles_trimmed = n_cycles
    
    # Central-difference velocity
    vel_norm = np.zeros_like(com_norm)
    vel_norm[:, 1:-1, :] = (com_norm[:, 2:, :] - com_norm[:, :-2, :]) / (2 * dt_actual)
    vel_norm[:, 0, :]     = (com_norm[:, 1, :] - com_norm[:, 0, :]) / dt_actual
    vel_norm[:, -1, :]    = (com_norm[:, -1, :] - com_norm[:, -2, :]) / dt_actual
    
    def _estimate_jacobian_and_max(S_pos, S_vel, return_jacobian=False):
        """Estimate Jacobian and maximum Floquet multiplier."""
        S = np.hstack([S_pos, S_vel])
        X = S[:-1, :]
        Y = S[1:,  :]
        
        # Find linearization point
        if use_fixed_point:
            linearization_point = find_fixed_point(S)
        else:
            linearization_point = S.mean(axis=0)
        
        dX = (X - linearization_point).T
        dY = (Y - linearization_point).T
        
        try:
            pinv_dX = np.linalg.pinv(dX, rcond=rcond)
            J = dY @ pinv_dX
        except np.linalg.LinAlgError:
            lam = rcond * np.trace(dX @ dX.T)
            J = dY @ dX.T @ np.linalg.inv(dX @ dX.T + lam * np.eye(dX.shape[0]))
        
        eigs = np.linalg.eigvals(J)
        mags = np.abs(eigs)
        
        # Check for neutral mode (should be close to 1)
        neutral_idx = np.argmin(np.abs(mags - 1.0))
        if np.abs(mags[neutral_idx] - 1.0) > 0.1:
            None
        
        
        if return_all_eigenvalues:
            sorted_idx = np.argsort(mags)[::-1]
            return eigs[sorted_idx], mags[sorted_idx], J if return_jacobian else None
        elif return_jacobian:
            return eigs, np.max(mags), J
        else:
            return eigs, np.max(mags)
    
    def _bootstrap_floquet(S_pos, S_vel, n_boot=100, ci=95.0):
        """Compute bootstrap confidence interval for max Floquet multiplier."""
        n_samples = S_pos.shape[0]
        max_lams = []
        
        for _ in range(n_boot):
            indices = resample(np.arange(n_samples), n_samples=n_samples)
            S_pos_boot = S_pos[indices]
            S_vel_boot = S_vel[indices]
            
            try:
                _, max_lam = _estimate_jacobian_and_max(S_pos_boot, S_vel_boot)
                max_lams.append(max_lam)
            except:
                continue
        
        if len(max_lams) > 0:
            max_lams = np.array(max_lams)
            ci_low = np.percentile(max_lams, (100 - ci) / 2)
            ci_high = np.percentile(max_lams, 100 - (100 - ci) / 2)
            return np.mean(max_lams), (ci_low, ci_high)
        else:
            return None, (None, None)
    
    # Compute results
    if not phase_dependent:
        S_pos = com_norm[:, 0, :]
        S_vel = vel_norm[:, 0, :]
        
        # Compute main result
        if return_all_eigenvalues:
            eigs, mags = _estimate_jacobian_and_max(S_pos, S_vel, return_jacobian=False)[:2]
            max_lam = mags[0]  # sorted by magnitude
        else:
            _, max_lam = _estimate_jacobian_and_max(S_pos, S_vel)
        
        # Bootstrap confidence interval
        mean_lam, (ci_low, ci_high) = _bootstrap_floquet(S_pos, S_vel, bootstrap_n, bootstrap_ci)
        
        # Poincaré plot
        if n_axes == 1:
            # For single axis analysis, plot in position-velocity space
            S = np.hstack([S_pos, S_vel])
            xk = S[:-1, 0]
            xk1 = S[1:, 0]
            
            axis_labels = {
                'x': ('Medio-lateral', 'ML'),
                'y': ('Anterior-posterior', 'AP'),
                'z': ('Vertical', 'Vert')
            }
            full_name, short_name = axis_labels[axes[0]]
            
            plt.figure(figsize=(6, 6))
            color = 'blue' if side_used == 'Right' else 'red'
            plt.scatter(xk, xk1, s=30, alpha=0.6, c=color, label=f'{side_used}→{side_used}')
            
            mn, mx = min(xk.min(), xk1.min()), max(xk.max(), xk1.max())
            plt.plot([mn, mx], [mn, mx], 'k--', lw=1, alpha=0.5)
            plt.xlabel(f'{short_name} position at strike k (m)')
            plt.ylabel(f'{short_name} position at strike k+1 (m)')
            plt.title(f'Poincaré Map ({full_name}, {side_used} side)\nMax |λ| = {max_lam:.3f} [{ci_low:.3f}, {ci_high:.3f}]')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.axis('equal')
            plt.show()
            plt.close()
        else:
            # For multi-axis analysis, plot using first axis
            S = np.hstack([S_pos, S_vel])
            axis_idx = 0  # Use first axis for visualization
            xk = S[:-1, axis_idx]
            xk1 = S[1:, axis_idx]
            
            plt.figure(figsize=(6, 6))
            color = 'blue' if side_used == 'Right' else 'red'
            plt.scatter(xk, xk1, s=30, alpha=0.6, c=color, label=f'{side_used}→{side_used}')
            
            mn, mx = min(xk.min(), xk1.min()), max(xk.max(), xk1.max())
            plt.plot([mn, mx], [mn, mx], 'k--', lw=1, alpha=0.5)
            
            state_dim_str = f"{2*n_axes}D"  # Dynamic dimension string
            plt.xlabel(f'{axes[0].upper()} position at strike k (m)')
            plt.ylabel(f'{axes[0].upper()} position at strike k+1 (m)')
            plt.title(f'Poincaré Map (Combined {state_dim_str}, {side_used} side)\nMax |λ| = {max_lam:.3f} [{ci_low:.3f}, {ci_high:.3f}]')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.axis('equal')
            plt.show()
            plt.close()
        
        result = {
            'max_floquet': float(max_lam),
            'confidence_interval': (ci_low, ci_high),
            'n_cycles_used': n_cycles_trimmed,
            'trimmed_cycles': trim_cycles,
            'side_used': side_used if use_same_side else 'both',
            'axes_analyzed': axes,
            'mean_stride_duration': mean_duration,
            'used_fixed_point': use_fixed_point
        }
        
        if return_all_eigenvalues:
            result['all_eigenvalues'] = eigs
            result['all_magnitudes'] = mags
        
        return result
    
    else:
        # Phase-dependent analysis
        max_vs_phi = np.zeros(M)
        ci_low_vs_phi = np.zeros(M)
        ci_high_vs_phi = np.zeros(M)
        
        for phi in range(M):
            S_pos = com_norm[:, phi, :]
            S_vel = vel_norm[:, phi, :]
            
            _, max_vs_phi[phi] = _estimate_jacobian_and_max(S_pos, S_vel)
            
            # Bootstrap for this phase
            _, (ci_low, ci_high) = _bootstrap_floquet(S_pos, S_vel, bootstrap_n, bootstrap_ci)
            ci_low_vs_phi[phi] = ci_low if ci_low is not None else max_vs_phi[phi]
            ci_high_vs_phi[phi] = ci_high if ci_high is not None else max_vs_phi[phi]
        
        phases = np.linspace(0, 100, M)
        
        plt.figure(figsize=(10, 6))
        plt.plot(phases, max_vs_phi, '-', linewidth=2, label=f'Mean ({side_used} side)')
        plt.fill_between(phases, ci_low_vs_phi, ci_high_vs_phi, alpha=0.3, label=f'{bootstrap_ci}% CI')
        plt.axhline(y=1.0, color='r', linestyle='--', label='Stability boundary')
        plt.xlabel('Gait phase (%)')
        plt.ylabel('Max |λ|')
        
        if n_axes == 1:
            axis_name = axes[0]
            axis_label = {'x': 'Medio-lateral', 'y': 'Anterior-posterior', 'z': 'Vertical'}[axis_name]
            plt.title(f'Phase-dependent Floquet ({axis_label}, {side_used} side, n={n_cycles_trimmed})')
        else:
            state_dim_str = f"{2*n_axes}D"
            plt.title(f'Phase-dependent Floquet (Combined {state_dim_str}, {side_used} side, n={n_cycles_trimmed})')
        
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()
        plt.close()
        
        return {
            'max_floquet': max_vs_phi,
            'confidence_interval': (ci_low_vs_phi, ci_high_vs_phi),
            'n_cycles_used': n_cycles_trimmed,
            'trimmed_cycles': trim_cycles,
            'side_used': side_used if use_same_side else 'both',
            'axes_analyzed': axes,
            'mean_stride_duration': mean_duration,
            'used_fixed_point': use_fixed_point
        }


def compute_floquet_per_axis(com_da: xr.DataArray,
                            events_df: pd.DataFrame,
                            channel: str = "COM",
                            detrend_y: bool = True,
                            **kwargs):
    """Compute Floquet multipliers for x, y, and z axes individually."""
    results = {}
    
    print("\n" + "="*60)
    print("COMPUTING PER-AXIS FLOQUET MULTIPLIERS")
    print("="*60)
    
    for axis, name in [('x', 'Medio-lateral'), ('y', 'Anterior-posterior'), ('z', 'Vertical')]:
        print(f"\nAnalyzing {name} ({axis}) axis...")
        
        # Only detrend if it's the y-axis (due to treadmill speed)
        should_detrend = detrend_y and (axis == 'y')
        
        axis_result = compute_floquet_from_events(
            com_da=com_da,
            events_df=events_df,
            channel=channel,
            axes=[axis],
            detrend_y=should_detrend,
            **kwargs
        )
        results[axis] = axis_result
    
    return results


def extract_normalized_data_with_filter(com_window, df_window, 
                                       use_same_side=True, 
                                       side_used='Right',
                                       trim_cycles=5,
                                       lowpass_cutoff=8.0,
                                       normalization_points=101,
                                       detrend_y=True):
    """Extract normalized and filtered COM data matching the Floquet analysis preprocessing."""
    # Detrend y-axis (due to treadmill speed)
    com_window_processed = com_window.copy()
    if detrend_y:
        if 'y' in com_window.axis.values:
            y_data = com_window.sel(channel="COM", axis='y').values
            time_points = np.arange(len(y_data))
            coeffs = np.polyfit(time_points, y_data, 1)
            trend = np.polyval(coeffs, time_points)
            y_detrended = y_data - trend
            com_window_processed.loc[dict(channel="COM", axis='y')] = y_detrended
    
    # Get foot strike events
    df_fs = df_window.query("label=='Foot Strike'")[['time','context']].sort_values('time')
    strike_times = df_fs['time'].values
    strike_context = df_fs['context'].values
    
    # Determine which strikes to use
    if use_same_side:
        side_indices = np.where(strike_context == side_used)[0]
        stride_starts = side_indices[:-1]
        stride_ends = side_indices[1:]
        
    else:
        stride_starts = np.arange(len(strike_times) - 1)
        stride_ends = stride_starts + 1
    
    # Build raw segments
    raw_segs = []
    raw_ts = []
    stride_durations = []
    com_data = com_window_processed.sel(channel="COM", axis=["x","y","z"])  # All three axes
    
    for i, (start_idx, end_idx) in enumerate(zip(stride_starts, stride_ends)):
        t0 = strike_times[start_idx]
        t1 = strike_times[end_idx]
        
        seg = com_data.sel(time=slice(t0, t1)).transpose('time','axis')
        t = seg.time.values
        
        if t.size < 10:
            continue
            
        raw_segs.append(seg.values)
        raw_ts.append(t)
        stride_durations.append(t1 - t0)
    
    # Calculate mean duration BEFORE trimming
    mean_duration = np.mean(stride_durations)
    
    # Apply low-pass filter to each segment
    filtered_segs = []
    for seg_vals, t in zip(raw_segs, raw_ts):
        fs = 1.0 / np.mean(np.diff(t))
        filtered_seg = apply_lowpass_filter(seg_vals, fs, lowpass_cutoff)
        filtered_segs.append(filtered_seg)
    
    # Time-normalize each cycle
    n_cycles = len(filtered_segs)
    M = normalization_points
    norm_t = np.linspace(0, 1, M)
    com_norm = np.zeros((n_cycles, M, 3))  
    
    for idx, (vals, t) in enumerate(zip(filtered_segs, raw_ts)):
        t_norm = (t - t[0])/(t[-1] - t[0])
        for ax in range(3):  # All 3 axes
            com_norm[idx, :, ax] = np.interp(norm_t, t_norm, vals[:, ax])
    
    # Trim cycles
    if trim_cycles > 0 and n_cycles > 2*trim_cycles:
        com_norm = com_norm[trim_cycles:-trim_cycles]
        print(f"Trimmed {trim_cycles} cycles from start/end")
    
    # Calculate velocities
    dt_actual = mean_duration / (M - 1)
    vel_norm = np.zeros_like(com_norm)
    vel_norm[:, 1:-1, :] = (com_norm[:, 2:, :] - com_norm[:, :-2, :]) / (2 * dt_actual)
    vel_norm[:, 0, :]     = (com_norm[:, 1, :] - com_norm[:, 0, :]) / dt_actual
    vel_norm[:, -1, :]    = (com_norm[:, -1, :] - com_norm[:, -2, :]) / dt_actual
    
    return com_norm, vel_norm, mean_duration

In [ ]:

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd


def get_foot_strike(events_path, foot='Right', start_time=20.0, end_time=None):
   """ Get first and last foot strike times within a specified time window """
   
   # load events from CSV or NetCDF file
   if events_path.suffix == '.csv':
       df_events = pd.read_csv(events_path)
   elif events_path.suffix == '.nc':
       events_ds = xr.open_dataset(events_path)
       events_da = events_ds["events"] if "events" in events_ds else events_ds
       df_events = events_da.to_dataframe().reset_index()
   else:
       raise ValueError(f"Unsupported file type: {events_path.suffix}")
   
   # filter for foot strikes within time window
   foot_strikes = df_events[
       (df_events['label'] == 'Foot Strike') & 
       (df_events['context'] == foot) &
       (df_events['time'] >= start_time) &
       (df_events['time'] <= end_time)
   ]
   
   # get first and last strike times
   first_strike = foot_strikes['time'].min() if len(foot_strikes) > 0 else None
   last_strike = foot_strikes['time'].max() if len(foot_strikes) > 0 else None
   
   return {
       'first_strike': first_strike,
       'last_strike': last_strike
   }


def get_average_stride_duration(events_path, foot='Right', start_time=20.0, end_time=None):
   """" Get average stride duration for a specified foot within a time window """

   # load events from file
   if events_path.suffix == '.csv':
       df_events = pd.read_csv(events_path)
   elif events_path.suffix == '.nc':
       events_ds = xr.open_dataset(events_path)
       events_da = events_ds["events"] if "events" in events_ds else events_ds
       df_events = events_da.to_dataframe().reset_index()
   else:
       raise ValueError(f"Unsupported file type: {events_path.suffix}")
   
   # filter and sort foot strikes
   foot_strikes = df_events[
       (df_events['label'] == 'Foot Strike') & 
       (df_events['context'] == foot) &
       (df_events['time'] >= start_time) &
       (df_events['time'] <= end_time)
   ].sort_values('time')
   
   # calculate stride durations
   strike_times = foot_strikes['time'].values
   # compute the time difference between consecutive strikes
   durations = np.diff(strike_times) 
   
   if len(durations) > 0:
       return {
           'mean_duration': np.mean(durations),
           'std_duration': np.std(durations),
           'num_strides': len(durations),
           'durations': durations
       }
   else:
       return {
           'mean_duration': np.nan,
           'std_duration': np.nan,
           'num_strides': 0,
           'durations': np.array([])
       }

def average_mutual_information(ts: np.ndarray, lag: int, bins: int = 64) -> float:
   """ Compute Average Mutual Information (AMI) for a time series to search for a given lag """
   # create equal-occupancy bin edges
   edges = np.quantile(ts, np.linspace(0, 1, bins+1))
   x, y = ts[:-lag], ts[lag:]
   # compute 2D histogram
   hist2d, _, _ = np.histogram2d(x, y, bins=[edges, edges])
   # normalize to get joint probability
   pxy = hist2d / np.sum(hist2d)
   # marginal probabilities
   px = np.sum(pxy, axis=1)
   py = np.sum(pxy, axis=0)
   # mutual information calculation
   mask = pxy > 0
   return np.sum(pxy[mask] * np.log(pxy[mask] / (px[:, None] * py[None, :])[mask]))


def optimal_lag(ts: np.ndarray,
               fs: float = 100.0,
               max_lag: int = 50,
               bins: int = 64) -> int: 
   """ Find optimal lag for time series using AMI method """

   # compute AMI for all lags
   ami_vals = [average_mutual_information(ts, lag, bins)
               for lag in range(1, max_lag + 1)]
   lags = np.arange(1, max_lag + 1)
   ami1 = ami_vals[0]

   # plot AMI curve - for visualization
   plt.figure(figsize=(5, 3))
   plt.plot(lags, ami_vals, '-o', label='AMI(τ)')
   plt.axhline(ami1 / np.e, color='r', linestyle='--', label='AMI(1)/e')
   plt.xlabel('Lag (samples)')
   plt.ylabel('AMI')
   plt.title('AMI vs. Lag')
   plt.grid(True)
   plt.legend()
   plt.tight_layout()
   plt.show()

   τ_max = min(max_lag, int(0.20 * fs)) 

   # can't form a "local minimum" search, 
   # if lag too low, so just skip to fallback
   if τ_max < 3:
       return 1

   # search for local minimum with constraints
   for i in range(1, τ_max - 1):  
       if ami_vals[i] <= ami_vals[i - 1] and ami_vals[i] <= ami_vals[i + 1]:
           τ_candidate = i + 1  
           # only accept it if AMI has fallen by ≥80%
           if τ_candidate < 5:
               if ami_vals[i] <= 0.80 * ami1:
                   return τ_candidate
               else:
                   continue
           return τ_candidate

   # fallback #1: first lag where AMI(τ) smaller than AMI(1)/e
   threshold = ami1 / np.e
   for i in range(τ_max): 
       if ami_vals[i] <= threshold:
           return i + 1

   # fallback #2: global minimum up to max_lag
   return int(np.argmin(ami_vals) + 1)

def compute_fnn_percentage(ts: np.ndarray, lag: int,
                           max_dim: int = 15, rtol: float = 0.15):
    """ Compute False Nearest Neighbors (FNN) percentage for a time series """

    N = len(ts)
    dims = np.arange(1, max_dim + 1)
    fnn_perc = np.zeros_like(dims, dtype=float)

    for idx, m in enumerate(dims):
        M = N - m * lag
        if M <= 0:
            fnn_perc[idx:] = np.nan
            break
        # build m-dimensional delay vectors
        embedded = np.vstack([ts[i * lag : i * lag + M] for i in range(m)]).T
        false = total = 0
        for i in range(M):
            # compute distances to all other points
            dists = np.linalg.norm(embedded - embedded[i], axis=1)
            dists[i] = np.inf
            # exclude temporally too-close points (Theiler window)
            dists[max(0, i - m * lag) : i + m * lag] = np.inf
            j = np.argmin(dists)
            if dists[j] == np.inf:
                continue
            d_extra = abs(ts[i + m * lag] - ts[j + m * lag])
            if d_extra / dists[j] > rtol:
                false += 1
            total += 1
        fnn_perc[idx] = false / total if total else np.nan

    return dims, fnn_perc

def optimal_embedding_dimension(ts: np.ndarray,
                               lag: int,
                               max_dim: int = 15,
                               rtol: float = 0.15) -> int:
   """ Find optimal embedding dimension using 
   False Nearest Neighbors (FNN) method """
   
   # compute FNN percentages
   dims, fnn = compute_fnn_percentage(ts, lag, max_dim, rtol)

   if len(dims) < 3 or np.all(np.isnan(fnn)):
       return max_dim
   
   # find elbow using second difference
   second_diff = np.diff(fnn, n=2)

   if np.all(np.isnan(second_diff)) or np.allclose(second_diff, 0, equal_nan=True):
       return max_dim

   # elbow at maximum curvature
   i_elbow = int(np.nanargmax(np.abs(second_diff)))
   m_elbow = i_elbow + 2

   # bounds checking
   if m_elbow < 1:
       return 1
   if m_elbow > max_dim:
       return max_dim
   return m_elbow


def compute_lyapunov_exponent(ts: np.ndarray, emb_dim: int, lag: int,
                              fs: float = 100.0, max_time: float = 5.0,
                              fit_start: float = 0.1, fit_end: float = 0.5,
                              plot_residuals: bool = True) -> float:
    """" Compute Lyapunov exponent from time series data, 
    using proper phase space reconstruction"""
    N = len(ts)
    M = N - (emb_dim - 1) * lag
    if M <= 0:
        return np.nan

    embedded = np.vstack([ts[i * lag : i * lag + M] for i in range(emb_dim)]).T
    min_sep = emb_dim * lag
    
    # calculate distance threshold for neighbor selection
    min_dist_threshold = np.std(embedded) * 0.01 
    
    divergence = []
    skipped_points = 0  # points skipped because no valid neighbors

    for i in range(M):
        dists = np.linalg.norm(embedded - embedded[i], axis=1)
        dists[i] = np.inf
        dists[max(0, i - min_sep) : i + min_sep] = np.inf
        
        # improved neighbor selection with distance threshold
        valid_neighbors = dists < min_dist_threshold
        if valid_neighbors.any():
            # find closest among valid neighbors
            valid_dists = dists.copy()
            valid_dists[~valid_neighbors] = np.inf
            j = np.argmin(valid_dists)
        else:
            # fallback to original approach if no neighbors within threshold
            j = np.argmin(dists)
            if dists[j] == np.inf:
                skipped_points += 1
                continue

        k_max = min(M - i, M - j, int(max_time * fs))
        for k in range(1, k_max):
            d = np.linalg.norm(embedded[i + k] - embedded[j + k])
            if d > 0:
                divergence.append((k / fs, np.log(d)))

    if not divergence:
        print(f"Warning: No valid divergence data found. Skipped {skipped_points} points.")
        return np.nan

    # print diagnostic info - for debugging/qualtiy check
    print(f"Distance threshold: {min_dist_threshold:.6f}")
    print(f"Skipped points due to no valid neighbors: {skipped_points}/{M}")
    print(f"Total divergence data points: {len(divergence)}")

    data = np.array(divergence)
    times, logs = data[:, 0], data[:, 1]

    # bin and average
    bins = np.linspace(0, max_time, 50)
    centers, means = [], []
    for b in range(len(bins) - 1):
        mask = (times >= bins[b]) & (times < bins[b + 1])
        if mask.any():
            centers.append((bins[b] + bins[b + 1]) / 2)
            means.append(logs[mask].mean())

    centers = np.array(centers)
    means = np.array(means)
    
    
    # plot divergence curve as visual inspection
    plt.figure(figsize=(10, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(centers, means, 'o-')
    plt.axvline(fit_start, color='r', linestyle='--', label=f'{fit_start} s')
    plt.axvline(fit_end, color='r', linestyle='--', label=f'{fit_end} s')
    plt.xlabel('Time (s)')
    plt.ylabel('Average ln(distance)')
    plt.title('Divergence curve')
    plt.grid(True)
    plt.legend()

    # fit only over fit_start–fit_end s (because is sLE)
    fit_mask = (centers >= fit_start) & (centers <= fit_end)
    if fit_mask.sum() < 2:
        print("Warning: Insufficient data points for reliable fit")
        plt.show()
        return np.nan

    slope, intercept = np.polyfit(centers[fit_mask], means[fit_mask], 1)
    
    # uncomment the following lines, if no R sq needed
    # calculate fit quality R sq
    fitted_values = slope * centers[fit_mask] + intercept
    r_squared = 1 - np.sum((means[fit_mask] - fitted_values)**2) / np.var(means[fit_mask]) if np.var(means[fit_mask]) > 0 else 0
    print(f"Fit quality (R²): {r_squared:.3f}")
    
    if r_squared < 0.5:
        print(f"Warning: Poor fit quality (R² = {r_squared:.3f})")
    
    # plot residuals if requested
    if plot_residuals:
        plt.subplot(1, 2, 2)
        # calculate residuals
        residuals = means[fit_mask] - fitted_values
        
        # plot residuals as quality check
        plt.scatter(centers[fit_mask], residuals, color='blue', alpha=0.6)
        plt.axhline(y=0, color='red', linestyle='--', label='Zero line')
        plt.xlabel('Time (s)')
        plt.ylabel('Residuals')
        plt.title(f'Residuals of linear fit ({fit_start:.2f}-{fit_end:.2f} s)')
        plt.grid(True)
        plt.legend()
        
        # add fit line to divergence plot - for visualization
        plt.subplot(1, 2, 1)
        plt.plot(centers[fit_mask], fitted_values, 'r-', linewidth=2, 
                 label=f'Linear fit: slope={slope:.3f}, R²={r_squared:.3f}')
        plt.legend()
    
    plt.tight_layout()
    plt.show()

    return slope

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

def compute_spatial_nsi(markers: xr.DataArray,
                       window_samples: int = 500,
                       fs: float = 100.0) -> dict:
   """  Compute Spatial Non-Stationarity Index (NSI) 
   for the Center of Mass (COM) marker"""
   
   # extract COM marker and reorganize dimensions
   com = markers.sel(channel='COM').transpose('time', 'axis')
   data = com.values.copy()  

   com_dt = xr.DataArray(data, coords=com.coords, dims=com.dims)

   # plot filtered COM traces for all axes - ok for nsi
   fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
   axis_names = ['x', 'y', 'z']
   axis_labels = ['ML (x)', 'AP (y)', 'VT (z)']
   
   for idx, (ax_name, ax_label) in enumerate(zip(axis_names, axis_labels)):
       if ax_name in com_dt.axis.values:
           axes[idx].plot(com_dt.sel(axis=ax_name).values, label=f"Filtered {ax_label}")
           axes[idx].set_ylabel(f"{ax_label} Position\n(detrended)")
           axes[idx].legend()
           axes[idx].grid(True)
   
   axes[0].set_title("High-Pass Filtered COM Traces (All Axes)")
   axes[-1].set_xlabel("Sample")
   plt.tight_layout()
   plt.show()

   # calculate nsi with detailed window statistics
   def nsi_detailed(ts: np.ndarray, w: int) -> tuple:
       n = len(ts)
       if n < w:
           return np.nan, []
       
       windows = []
       window_means = []
       window_stds = []
       
       # process non-overlapping windows
       for i in range(0, n - w + 1, w):
           window_data = ts[i:i+w]
           windows.append((i, i+w))
           window_means.append(np.mean(window_data))
           window_stds.append(np.std(window_data, ddof=1))
       
       # actual calculation
       overall_std = np.std(ts, ddof=1)
       if overall_std == 0:
           return np.nan, []
       
       nsi_value = np.std(window_means, ddof=1) / overall_std
       
       return nsi_value, list(zip(windows, window_means, window_stds))

   # prints
   N = com_dt.sizes['time']
   M = (N - window_samples) // window_samples + 1
   print(f"COM series length: {N} samples ({N/fs:.2f} s)")
   print(f"Window size:       {window_samples} samples ({window_samples/fs:.2f} s)")
   print(f"Number of windows: {M}")
   print("-" * 80)

   # compute nsi for each axis
   results = {}
   detailed_results = {}
   
   for ax in com_dt.axis.values:
       arr = com_dt.sel(axis=ax).values
       nsi_val, window_details = nsi_detailed(arr, window_samples)
       results[ax] = nsi_val
       detailed_results[ax] = window_details

       # print window statistics
       print(f"\nAxis: {ax}")
       print(f"Overall NSI: {nsi_val:.4f}")
       print(f"{'Window':<15} {'Time Range (s)':<20} {'Mean':<12} {'Std':<12}")
       print("-" * 60)
       
       for idx, ((start, end), mean, std) in enumerate(window_details):
           time_start = start / fs
           time_end = end / fs
           print(f"Window {idx+1:<8} {time_start:6.2f} - {time_end:6.2f} s    {mean:8.4f}    {std:8.4f}")
  
   # create detailed visualization
   fig = plt.figure(figsize=(16, 12))
   gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
   
   axis_names = ['x', 'y', 'z']
   axis_labels = ['ML (x)', 'AP (y)', 'VT (z)']
   colors = ['blue', 'green', 'red']
   
   for idx, (ax_name, ax_label, color) in enumerate(zip(axis_names, axis_labels, colors)):
       if ax_name in com_dt.axis.values and ax_name in detailed_results:
           # plot time series with windows
           ax1 = fig.add_subplot(gs[idx, 0])
           time_array = np.arange(len(com_dt.sel(axis=ax_name))) / fs
           ax1.plot(time_array, com_dt.sel(axis=ax_name).values, 'k-', alpha=0.5, linewidth=0.5)
           
           window_data = detailed_results[ax_name]
           for win_idx, ((start, end), mean, std) in enumerate(window_data):
               t_start = start / fs
               t_end = end / fs
               # shade windows
               ax1.axvspan(t_start, t_end, alpha=0.2, color=color if win_idx % 2 == 0 else 'gray')
               # show window means
               ax1.hlines(mean, t_start, t_end, colors=color, linewidth=2)
               # label windows
               ax1.text((t_start + t_end) / 2, ax1.get_ylim()[1] * 0.9, f'W{win_idx+1}', 
                       ha='center', va='top', fontsize=8)
           
           ax1.set_ylabel(f'{ax_label} Position')
           ax1.set_title(f'{ax_label} Time Series with Windows')
           ax1.grid(True, alpha=0.3)
           if idx == 2:
               ax1.set_xlabel('Time (s)')
          
           # plot window statistics
           ax2 = fig.add_subplot(gs[idx, 1])
           window_indices = list(range(1, len(window_data) + 1))
           window_means = [w[1] for w in window_data]
           window_stds = [w[2] for w in window_data]
           
           ax2.errorbar(window_indices, window_means, yerr=window_stds, 
                       fmt='o-', color=color, capsize=5, capthick=2,
                       label='Mean ± Std')
           ax2.axhline(y=np.mean(window_means), color='black', linestyle='--', 
                      alpha=0.5, label='Overall mean')
           
           ax2.set_ylabel('Value')
           ax2.set_title(f'{ax_label} Window Statistics')
           ax2.grid(True, alpha=0.3)
           ax2.legend()
           if idx == 2:
               ax2.set_xlabel('Window Number')
           
           # plot deviations from overall mean
           ax3 = fig.add_subplot(gs[idx, 2])
           overall_mean = np.mean(com_dt.sel(axis=ax_name).values)
           deviations = [w[1] - overall_mean for w in window_data]
           
           ax3.bar(window_indices, deviations, color=color, alpha=0.7)
           ax3.axhline(y=0, color='black', linestyle='-', linewidth=1)
           ax3.set_ylabel('Deviation from Overall Mean')
           ax3.set_title(f'{ax_label} Window Mean Deviations')
           ax3.grid(True, alpha=0.3)
           
           # add NSI value
           ax3.text(0.98, 0.95, f'NSI = {results.get(ax_name, np.nan):.4f}', 
                   transform=ax3.transAxes, 
                   verticalalignment='top', horizontalalignment='right',
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
           
           if idx == 2:
               ax3.set_xlabel('Window Number')
   
   fig.suptitle(f'NSI Window Analysis (Window size: {window_samples/fs:.1f}s, Total: {N/fs:.1f}s)', 
                fontsize=14)
   plt.tight_layout()
   plt.show()
   
   # summary plots just for window - just check
   fig2, ax = plt.subplots(figsize=(12, 6))
   
   for ax_name, color in zip(axis_names, colors):
       if ax_name in detailed_results:
           window_data = detailed_results[ax_name]
           if window_data:
               window_centers = [(w[0][0] + w[0][1]) / 2 / fs for w in window_data]
               window_means = [w[1] for w in window_data]
               
               ax.plot(window_centers, window_means, 'o-', color=color, 
                      label=f'{ax_name.upper()} (NSI={results.get(ax_name, np.nan):.4f})',
                      markersize=8, linewidth=2)
   
   ax.set_xlabel('Time (s)')
   ax.set_ylabel('Window Mean Position')
   ax.set_title('Window Means Comparison Across All Axes')
   ax.grid(True, alpha=0.3)
   ax.legend()
   plt.tight_layout()
   plt.show()

   return results, detailed_results


def compute_temporal_nsi_from_events(df_events: pd.DataFrame,
                                    strides_per_window: int = 5,
                                    trim_strides: int = 0,
                                    outlier_thresh: float = 3.0) -> float:
   """ Compute Temporal Non-Stationarity Index (NSI)"""
   
   # filter for Foot Strike events and sort by time
   df_fs = (df_events[df_events['label'] == 'Foot Strike']
            .sort_values('time'))

   # calculate time intervals between consecutive foot strikes
   intervals = df_fs['time'].diff().dropna().values

   # remove startup/shutdown strides from both ends - this is needed (adjustable)
   if len(intervals) > 2 * trim_strides:
       intervals = intervals[trim_strides:-trim_strides]

   # remove outliers beyond threshold * standard deviations
   mu, sigma = intervals.mean(), intervals.std(ddof=1)
   mask = np.abs(intervals - mu) <= outlier_thresh * sigma
   intervals = intervals[mask]

   # calculate temp nsi using block averaging
   def nsi(ts, k):
       n = len(ts)
       if n < k or ts.std(ddof=1) == 0:
           return np.nan
       # create non-overlapping blocks of k strides and compute their means
       block_means = [ts[i:i+k].mean() for i in range(0, n-k+1, k)]
       # nsi = variability of block means / overall variability
       return np.std(block_means, ddof=1) / ts.std(ddof=1)

   return nsi(intervals, strides_per_window)

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy.linalg import eig
from scipy.interpolate import interp1d
from skfda import FDataGrid
from skfda.preprocessing.dim_reduction import FPCA
from skfda.representation.basis import BSplineBasis as BSpline
from typing import Optional, Dict, Tuple, List

sns.set_theme(style="whitegrid", font_scale=1.1)

class EventAlignedFPCA_MSM_RMCE:
    """
    Event-aligned MSM for gait stability using FPCA + K-means:
    each stride (event-to-event) → functional representation → FPCA → K-means → Stable/Marginal/Unstable.
    """
    def __init__(self,
                 patient_id: str,
                 n_states: int = 3,
                 n_grid_points: int = 100,  # Number of points to interpolate each stride to
                 n_components: int = 7,
                 n_basis: int = 15):
        self.patient_id        = patient_id
        self.n_states          = n_states
        self.n_grid_points     = n_grid_points  # For interpolation
        self.n_components      = n_components
        self.n_basis           = n_basis
        self.scaler            = StandardScaler()
        self.metadata          = {}
        self.fpca              = None
        self.kmeans            = None
        self.transition_matrix = None
        self.stationary_dist   = None
        self.state_labels      = {0:"Stable", 1:"Marginally Stable", 2:"Unstable"}
        self.metrics_history   = None
        self.fdata             = None
        self.fpca_scores       = None
        self.stride_info       = None  # Store stride timing information
        self.validation_metrics = {}  # Store validation metrics

    def compute_event_aligned_functional_data(self, signal: np.ndarray, events: np.ndarray, fs: float) -> FDataGrid:
        """
        Convert event-aligned segments into functional data objects.
        Each segment is from one event to the next (e.g., foot strike to foot strike).
        """
        # Collect stride segments
        segments = []
        stride_times = []
        stride_durations = []
        valid_strides = []
        
        for i in range(len(events) - 1):
            start_time = events[i]
            end_time = events[i + 1]
            start_idx = int(start_time * fs)
            end_idx = int(end_time * fs)
            
            # Check bounds
            if start_idx >= 0 and end_idx <= len(signal):
                segment = signal[start_idx:end_idx]
                
                # Skip very short or very long strides (outliers)
                stride_duration = end_time - start_time
                if 0.5 < stride_duration < 2.0:  # Reasonable stride duration range
                    segments.append(segment)
                    stride_times.append(start_time)
                    stride_durations.append(stride_duration)
                    valid_strides.append(i)
        
        # Interpolate all segments to common grid
        interpolated_segments = []
        common_grid = np.linspace(0, 1, self.n_grid_points)
        
        for segment in segments:
            # Create interpolation function
            original_grid = np.linspace(0, 1, len(segment))
            f = interp1d(original_grid, segment, kind='cubic', fill_value='extrapolate')
            
            # Interpolate to common grid
            interpolated_segment = f(common_grid)
            interpolated_segments.append(interpolated_segment)
        
        # Convert to FDataGrid
        self.fdata = FDataGrid(
            data_matrix=np.array(interpolated_segments),
            grid_points=common_grid
        )
        
        # Store metadata - later for visuals and analysis
        self.metrics_history = pd.DataFrame({
            'stride_idx': range(len(segments)),
            'original_event_idx': valid_strides,
            'time': stride_times,
            'duration': stride_durations
        })
        
        # Store stride info for later use
        self.stride_info = {
            'events': events,
            'valid_strides': valid_strides,
            'fs': fs
        }
        
        print(f"Created functional data from {len(segments)} valid strides")
        print(f"Average stride duration: {np.mean(stride_durations):.3f}s ± {np.std(stride_durations):.3f}s")
        
        return self.fdata

    def apply_fpca(self) -> np.ndarray:
        """Apply Functional PCA to extract principal components."""
        try:
            # smooth the functional data using B-spline basis
            basis = BSpline(n_basis=self.n_basis, domain_range=(0, 1))
            
            smooth_fdata = self.fdata.to_basis(basis)
            
            # apply FPCA to smoothed data
            self.fpca = FPCA(n_components=self.n_components)
            self.fpca_scores = self.fpca.fit_transform(smooth_fdata)
        except:
            # If smoothing fails, apply FPCA directly to the discrete data
            print("Note: B-spline smoothing failed, using direct FPCA on discrete data")
            self.fpca = FPCA(n_components=self.n_components)
            self.fpca_scores = self.fpca.fit_transform(self.fdata)
        
        # add FPCA scores to metrics history
        for i in range(self.n_components):
            self.metrics_history[f'fpca_{i}'] = self.fpca_scores[:, i]
        
        # compute FPCA reconstruction error
        self._compute_fpca_reconstruction_error()
        
        return self.fpca_scores


    def _compute_fpca_reconstruction_error(self):
        """Compute reconstruction error from FPCA, plus NRMSE wrt range & std."""
        try:
            recon = self.fpca.inverse_transform(self.fpca_scores)
            if not isinstance(recon, FDataGrid):
                recon = recon.to_grid(self.fdata.grid_points)

            orig = self.fdata.data_matrix.squeeze()  # (n_strides, n_grid)
            rec  = recon.data_matrix.squeeze()

            # 1) MSE & RMSE
            mse  = np.mean((orig - rec)**2)
            rmse = np.sqrt(mse)
            self.validation_metrics['fpca_reconstruction_mse'] = mse
            self.validation_metrics['fpca_reconstruction_rmse'] = rmse

            # 2) Signal range & std
            sig_min   = orig.min()
            sig_max   = orig.max()
            sig_range = sig_max - sig_min
            sig_std   = orig.std()
            self.validation_metrics.update({
                'signal_min': sig_min,
                'signal_max': sig_max,
                'signal_range': sig_range,
                'signal_std': sig_std,
            })

            # 3) Normalized errors
            nrmse_range = rmse / sig_range if sig_range>0 else np.nan
            nrmse_std   = rmse / sig_std   if sig_std>0   else np.nan
            self.validation_metrics['nrmse_range'] = nrmse_range
            self.validation_metrics['nrmse_std']   = nrmse_std

            print(f"FPCA reconstruction MSE: {mse:.4f}, RMSE: {rmse:.4f}")
            print(f"Signal range: {sig_range:.4f}, σ: {sig_std:.4f}")
            print(f"NRMSE (range): {nrmse_range:.2%},  NRMSE (σ): {nrmse_std:.2%}")

        except Exception as e:
            print(f"Errror: {e}")
            self.validation_metrics['fpca_reconstruction_mse'] = np.nan
    
    
    def _compute_kmeans_variance_explained(self):
        """ Compute variance explained by K-means clustering on FPCA scores."""
        # Pull out the FPCA scores 
        X = self.fpca_scores[:, :self.n_components]
        labels = self.metrics_history['state'].values
        
        # 1) global mean & TSS
        global_mean = X.mean(axis=0)
        TSS = np.sum((X - global_mean) ** 2)
        
        # 2) within‑cluster sum of squares (WSS)
        WSS = 0.0
        for k in range(self.n_states):
            mask = labels == k
            if mask.sum() == 0:
                continue
            cluster_data = X[mask]
            cluster_mean = cluster_data.mean(axis=0)
            WSS += np.sum((cluster_data - cluster_mean) ** 2)
        
        # 3) fraction explained
        R2 = 1 - WSS / TSS if TSS > 0 else 0.0
        self.validation_metrics['fpca_variance_explained_by_clustering'] = R2
        print(f"FPCA‑KMeans variance explained: {R2:.1%}")

    def identify_stability_states(self) -> np.ndarray:
        """Apply K-means clustering to FPCA scores to identify states."""
        
        # Standardize FPCA scores
        X = self.scaler.fit_transform(self.fpca_scores[:,:self.n_components])  
        
        # Apply K-means
        self.kmeans = KMeans(n_clusters=self.n_states, random_state=42)
        raw_states = self.kmeans.fit_predict(X)
        
        # Order states by distance from origin (stability proxy)
        cluster_centers = self.kmeans.cluster_centers_
        norms = np.linalg.norm(cluster_centers, axis=1)
        order = np.argsort(norms)
        
        # Remap states to ensure 0=stable, 2=unstable
        state_mapping = {old: new for new, old in enumerate(order)}
        states = np.array([state_mapping[s] for s in raw_states])
        
        self.metrics_history['state'] = states
        
        # Compute clustering validation metric
        self._compute_clustering_silhouette_score(X, states)
        self.metrics_history['state'] = states
        
        
        self._compute_clustering_silhouette_score(X, states)
       
        self._compute_kmeans_variance_explained()
    
        return states

    def _compute_clustering_silhouette_score(self, X: np.ndarray, labels: np.ndarray):
        """Compute silhouette score for clustering validation."""
        
        try:
            # Compute silhouette score
            sil_score = silhouette_score(X, labels)
            self.validation_metrics['clustering_silhouette_score'] = sil_score
            
            print(f"Clustering silhouette score: {sil_score:.3f}")
            
            # Interpretation
            if sil_score > 0.7:
                interpretation = "Strong clustering"
            elif sil_score > 0.5:
                interpretation = "Reasonable clustering"
            elif sil_score > 0.25:
                interpretation = "Weak clustering"
            else:
                interpretation = "Poor clustering"
            
            print(f"Clustering quality: {interpretation}")
            
        except Exception as e:
            print(f"Warning: Could not compute silhouette score: {e}")
            self.validation_metrics['clustering_silhouette_score'] = np.nan

    def compute_transition_matrix(self, states: np.ndarray) -> np.ndarray:
        """Compute state transition probabilities."""

        T = np.zeros((self.n_states, self.n_states))
        for a, b in zip(states[:-1], states[1:]):
            T[a, b] += 1
        
        # Normalize rows
        row_sums = T.sum(axis=1)
        row_sums[row_sums == 0] = 1
        T = T / row_sums[:, None]
        
        self.transition_matrix = T
        return T

    def compute_stationary_distribution(self) -> np.ndarray:
        """Compute stationary distribution of the Markov chain."""

        vals, vecs = eig(self.transition_matrix.T)
        idx = np.argmin(np.abs(vals - 1))
        pi = np.real(vecs[:, idx])
        pi = pi / pi.sum()
        self.stationary_dist = np.abs(pi)
        return self.stationary_dist

    def compute_mean_dwell_times(self) -> np.ndarray:
        """Compute mean dwell time in each state."""

        τ = np.zeros(self.n_states)
        for i in range(self.n_states):
            p = self.transition_matrix[i, i]
            τ[i] = np.inf if p >= 1 else 1 / (1 - p)
        return τ

    def compute_stability_metrics(self) -> dict:
        """Compute overall stability metrics."""

        df = self.metrics_history
        out = {'patient_id': self.patient_id}
        
        # State fractions
        for i in range(self.n_states):
            out[f'frac_state_{i}'] = (df.state == i).mean()
        
        # Mean FPCA scores per state
        for i in range(self.n_states):
            state_data = df[df.state == i]
            for j in range(self.n_components):
                out[f'mean_fpca_{j}_state_{i}'] = state_data[f'fpca_{j}'].mean()
        
        # Mean stride duration per state
        for i in range(self.n_states):
            state_data = df[df.state == i]
            out[f'mean_duration_state_{i}'] = state_data['duration'].mean()
        
        # Add validation metrics
        out.update(self.validation_metrics)
        
        return out

    def visualize_patient_analysis(self):
        """2×2 summary figure with FPCA results and transitions."""
        df = self.metrics_history.copy()
        names = list(self.state_labels.values())
        colors = sns.color_palette(n_colors=self.n_states)
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        # 1) FPCA explained variance
        if self.fpca is not None and hasattr(self.fpca, 'explained_variance_ratio_'):
            explained_var = self.fpca.explained_variance_ratio_
            pc_labels = [f'PC{i+1}' for i in range(len(explained_var))]
            axes[0, 0].bar(pc_labels, explained_var, color='skyblue')
            axes[0, 0].set_title('FPCA Explained Variance Ratio')
            axes[0, 0].set_ylabel('Variance Explained')
            for i, v in enumerate(explained_var):
                axes[0, 0].text(i, v + 0.01, f'{v:.1%}', ha='center')
                
        # 2) Transition matrix heatmap
        sns.heatmap(self.transition_matrix,
                    annot=True, fmt=".2f",
                    xticklabels=names, yticklabels=names,
                    cmap="Blues", ax=axes[0, 1])
        axes[0, 1].set_title("Transition Probabilities")
        
        # Add silhouette score
        if 'clustering_silhouette_score' in self.validation_metrics:
            sil_score = self.validation_metrics['clustering_silhouette_score']
            axes[0, 1].text(0.02, 0.98, f'Silhouette Score: {sil_score:.3f}', 
                           transform=axes[0, 1].transAxes, va='top',
                           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        # 3) FPCA scores scatter (first 2 PCs)
        if self.n_components >= 2 and 'fpca_0' in df.columns and 'fpca_1' in df.columns:
            for i, (state, name) in enumerate(self.state_labels.items()):
                mask = df.state == state
                axes[1, 0].scatter(df.loc[mask, 'fpca_0'], 
                                 df.loc[mask, 'fpca_1'],
                                 label=name, color=colors[i], alpha=0.6)
            axes[1, 0].set_xlabel('FPCA Component 1')
            axes[1, 0].set_ylabel('FPCA Component 2')
            axes[1, 0].set_title('State Distribution in FPCA Space')
            axes[1, 0].legend()
        
        # 4) Stride duration by state
        # More informative than stationary distribution for event-aligned data
        duration_data = []
        for state, name in self.state_labels.items():
            durations = df[df.state == state]['duration'].values
            duration_data.extend([{'State': name, 'Duration': d} for d in durations])
        
        duration_df = pd.DataFrame(duration_data)
        # sns.boxplot(data=duration_df, x='State', y='Duration', palette=colors, ax=axes[1, 1])
        sns.boxplot(
            data=duration_df,
            x='State',
            y='Duration',
            hue='State',       
            palette=colors,
            dodge=False,        
            ax=axes[1, 1]
                )   
        axes[1, 1].set_title('Stride Duration by State')
        axes[1, 1].set_ylabel('Duration (s)')
        
        plt.tight_layout()
        plt.show()

    def visualize_functional_modes(self):
        """Visualize the functional principal components."""
        if self.fpca is None:
            return
        
        if not hasattr(self.fpca, 'mean_') or not hasattr(self.fpca, 'components_'):
            print("FPCA components not available for visualization")
            return
        
        fig, axes = plt.subplots(1, min(3, self.n_components), 
                                figsize=(5*min(3, self.n_components), 4))
        
        if self.n_components == 1:
            axes = [axes]
        
        # Create evaluation grid
        grid = np.linspace(0, 1, 100)
        
        try:
            # Get mean function and components
            mean_func = self.fpca.mean_
            
            # Evaluate mean function on grid
            mean_values = mean_func(grid).flatten()
            
            for i in range(min(3, self.n_components)):
                # Get principal component
                pc = self.fpca.components_[i]
                pc_values = pc(grid).flatten()
                
                # Plot mean plus PC
                axes[i].plot(grid, mean_values, 'k-', 
                            label='Mean', linewidth=2)
                
                # Scale by explained variance if available
                if hasattr(self.fpca, 'explained_variance_'):
                    scale = np.sqrt(self.fpca.explained_variance_[i])
                else:
                    scale = 1.0
                    
                axes[i].plot(grid, mean_values + 2*scale*pc_values, 
                            'b--', label='+2 SD', alpha=0.7)
                axes[i].plot(grid, mean_values - 2*scale*pc_values, 
                            'r--', label='-2 SD', alpha=0.7)
                
                if hasattr(self.fpca, 'explained_variance_ratio_'):
                    axes[i].set_title(f'PC{i+1} ({self.fpca.explained_variance_ratio_[i]:.1%} var)')
                else:
                    axes[i].set_title(f'PC{i+1}')
                    
                axes[i].set_xlabel('Normalized Stride Time')
                axes[i].legend()
        except Exception as e:
            print(f"Error visualizing functional modes: {e}")
            for ax in axes:
                ax.text(0.5, 0.5, 'Functional modes\nnot available', 
                       ha='center', va='center', transform=ax.transAxes)
        
        plt.tight_layout()
        plt.show()

    def visualize_stride_alignment(self, signal: np.ndarray, max_strides: int = 20):
        """Visualize how strides are aligned and colored by state."""
        if self.stride_info is None:
            return
        
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
        
        colors = sns.color_palette(n_colors=self.n_states)
        events = self.stride_info['events']
        fs = self.stride_info['fs']
        
        # Original signal with event markers and state colors
        time_axis = np.arange(len(signal)) / fs
        ax1.plot(time_axis, signal, 'k-', alpha=0.5, linewidth=0.5)
        
        # Add stride coloring
        for idx, stride_idx in enumerate(self.stride_info['valid_strides'][:max_strides]):
            if stride_idx < len(events) - 1:
                start_time = events[stride_idx]
                end_time = events[stride_idx + 1]
                state = int(self.metrics_history.iloc[idx]['state'])  # Convert to int
                ax1.axvspan(start_time, end_time, color=colors[state], alpha=0.3)
        
        # Add event markers
        ax1.scatter(events, np.interp(events, time_axis, signal), 
                   color='red', s=50, zorder=5, label='Foot Strikes')
        ax1.set_xlabel('Time (s)')
        ax1.set_ylabel('Signal Value')
        ax1.set_title('Original Signal with Event-Aligned State Classification')
        ax1.legend()
        
        # Overlaid normalized strides colored by state
        grid = np.linspace(0, 1, self.n_grid_points)
        for i in range(min(max_strides, len(self.fdata))):
            state = int(self.metrics_history.iloc[i]['state'])  # Convert to int
            ax2.plot(grid, self.fdata[i].data_matrix[0], 
                    color=colors[state], alpha=0.5, linewidth=1)
        
        # Add mean curves per state
        for state, name in self.state_labels.items():
            state_mask = self.metrics_history['state'] == state
            if state_mask.sum() > 0:
                state_curves = self.fdata[state_mask].data_matrix
                mean_curve = np.mean(state_curves, axis=0)
                ax2.plot(grid, mean_curve, color=colors[state], 
                        linewidth=3, label=name)
        
        ax2.set_xlabel('Normalized Stride Time (0=strike, 1=next strike)')
        ax2.set_ylabel('Signal Value')
        ax2.set_title('Event-Aligned Strides by State')
        ax2.legend()
        
        plt.tight_layout()
        plt.show()

    def fit(self, signal: np.ndarray, events: np.ndarray, fs: float) -> 'EventAlignedFPCA_MSM_RMCE':
        """
        Complete pipeline: signal + events → functional data → FPCA → clustering → MSM.
        """
        print(f"Processing patient {self.patient_id} with event-aligned strides...")
        
        # 1. Create functional data from event-aligned segments
        print("Creating event-aligned functional data...")
        self.compute_event_aligned_functional_data(signal, events, fs)
        
        # 2. Apply FPCA
        print(f"Applying FPCA with {self.n_components} components...")
        self.apply_fpca()
        
        # 3. Identify states via K-means
        print("Identifying stability states via K-means...")
        states = self.identify_stability_states()
        
        # 4. Build Markov model
        print("Computing transition matrix...")
        self.compute_transition_matrix(states)
        
        # 5. Compute stationary distribution
        print("Computing stationary distribution...")
        self.compute_stationary_distribution()
             
        metrics = self.compute_stability_metrics()
        print(f"\nStability metrics for {self.patient_id}:")
        for key, value in metrics.items():
            if key == 'patient_id':
                continue
            # if it's an array, summarize it rather than try to format directly
            if isinstance(value, np.ndarray):
                print(f"  {key}: array, n={value.size}, mean={value.mean():.3f}, std={value.std():.3f}")
            else:
                print(f"  {key}: {value:.3f}")

        # Print validation summary
        print(f"\nValidation Summary:")
       
        if 'fpca_reconstruction_rmse' in self.validation_metrics:
            rmse = self.validation_metrics['fpca_reconstruction_rmse']
            nrmse_r = self.validation_metrics.get('nrmse_range', np.nan)
            nrmse_s = self.validation_metrics.get('nrmse_std',   np.nan)
            print(f"  FPCA reconstruction → RMSE = {rmse:.4f}  "
            f"(NRMSE range = {nrmse_r:.2%}, NRMSE σ = {nrmse_s:.2%})")
        sil = self.validation_metrics.get('clustering_silhouette_score', None)
        if sil is not None and np.isscalar(sil):
            print(f"  Clustering quality: Silhouette = {sil:.3f}")

        return self
    def plot_fpca_reconstruction_error_distribution(self, bins: int = 20):
        """
        Histogram of per‑stride FPCA reconstruction MSE to spot outliers.
        """
        errors = self.validation_metrics.get('fpca_reconstruction_mse_per_stride')
        if errors is None:
            print("No per‑stride MSE data available.")
            return

        plt.figure(figsize=(8, 4))
        sns.histplot(errors, bins=bins, kde=True)
        plt.title('Per‑Stride FPCA Reconstruction MSE')
        plt.xlabel('Reconstruction MSE')
        plt.ylabel('Count')
        plt.tight_layout()
        plt.show()


    def plot_silhouette_samples(self):
        """
        Scatter of silhouette coefficient per stride.
        """
        from sklearn.metrics import silhouette_samples
        from sklearn.preprocessing import StandardScaler

        # Rescale all PCs afresh
        X = StandardScaler().fit_transform(self.fpca_scores)
        labels = self.metrics_history['state'].values
        sil_vals = silhouette_samples(X, labels)

        self.validation_metrics['silhouette_samples'] = sil_vals

        plt.figure(figsize=(10, 5))
        palette = sns.color_palette(n_colors=self.n_states)
        sns.scatterplot(
            x=np.arange(len(sil_vals)),
            y=sil_vals,
            hue=labels,
            palette=palette,
            legend='brief',
            alpha=0.7
        )
        plt.axhline(sil_vals.mean(), color='k', linestyle='--',
                    label=f'Mean = {sil_vals.mean():.3f}')
        plt.title('Silhouette Coefficient per Stride')
        plt.xlabel('Stride Index')
        plt.ylabel('Silhouette Coefficient')
        plt.legend(title='State')
        plt.tight_layout()
        plt.show()


    def plot_silhouette_analysis(self):
        """
        Classic silhouette‑analysis bar plot by cluster.
        """
        from sklearn.metrics import silhouette_samples, silhouette_score
        from sklearn.preprocessing import StandardScaler
        import matplotlib.cm as cm

        X = StandardScaler().fit_transform(self.fpca_scores)
        y = self.metrics_history['state'].values
        sil_vals = silhouette_samples(X, y)
        sil_avg  = silhouette_score(X, y)

        fig, ax1 = plt.subplots(figsize=(8, 5))
        y_lower = 10
        for i in range(self.n_states):
            ith_sil = np.sort(sil_vals[y == i])
            size_i  = ith_sil.shape[0]
            if size_i == 0:
                continue
            y_upper = y_lower + size_i
            color   = cm.nipy_spectral(float(i) / self.n_states)
            ax1.fill_betweenx(np.arange(y_lower, y_upper),
                              0, ith_sil,
                              facecolor=color, edgecolor=color, alpha=0.7)
            ax1.text(-0.05, y_lower + 0.5 * size_i, f"State {i}")
            y_lower = y_upper + 10

        ax1.axvline(sil_avg, color="red", linestyle="--",
                    label=f'Average silhouette = {sil_avg:.3f}')
        ax1.set_title("Silhouette Analysis per State")
        ax1.set_xlabel("Silhouette Coefficient")
        ax1.set_ylabel("Cluster")
        ax1.set_yticks([])
        ax1.legend(loc='upper right')
        plt.tight_layout()
        plt.show()

        
    def plot_all_feature_correlations(self):
        # 1) gather metadata columns
        meta_cols = ['stride_idx', 'time', 'duration']
        
        # 2) FPCA score columns
        fpca_cols = [f"fpca_{i}" for i in range(self.n_components)]
        
        # 3) validation metric columns (filter numeric entries)
        val_cols = [k for k, v in self.validation_metrics.items() 
                    if isinstance(v, (int, float, np.floating, np.integer))]
        
        # 4) combine and drop any that aren't in metrics_history
        all_cols = [c for c in (meta_cols + fpca_cols + val_cols) 
                    if c in self.metrics_history.columns]
        
        # build the DataFrame
        df = self.metrics_history[all_cols].copy()
        
        # if some validation metrics are arrays, skip them
        # (we only plot scalar metrics)
        
        # 5) compute correlation matrix
        corr = df.corr()
        
        # 6) plot heatmap
        plt.figure(figsize=(10, 8))
        sns.heatmap(
            corr,
            annot=True,
            fmt=".2f",
            cmap="vlag",
            center=0,
            square=True,
            cbar_kws={"shrink": .8},
            linewidths=0.5
        )
        plt.title(f"Patient {self.patient_id} — All Feature Correlations")
        plt.xticks(rotation=45, ha="right")
        plt.yticks(rotation=0)
        plt.tight_layout()
        plt.show()

        
    def plot_features_vs_clusters(self):
        # 1) pick numeric features (e.g., duration, FPCA scores, RMSE, silhouette…)
        meta_cols  = ['duration']
        fpca_cols  = [f"fpca_{i}" for i in range(self.n_components)]
        val_cols   = [k for k, v in self.validation_metrics.items() 
                    if isinstance(v, (int, float, np.integer, np.floating))]
        feat_cols  = [c for c in (meta_cols + fpca_cols + val_cols) 
                    if c in self.metrics_history.columns]
        
        df_feats   = self.metrics_history[feat_cols].copy()
        
        # 2) one‑hot encode the cluster labels
        df_states  = pd.get_dummies(self.metrics_history['state'], 
                                    prefix='state')
        
        # 3) combine
        df_all     = pd.concat([df_feats, df_states], axis=1)
        
        # 4) compute correlation
        corr = df_all.corr()
        
        # 5) plot
        plt.figure(figsize=(10,8))
        sns.heatmap(
            corr.loc[feat_cols, df_states.columns],   # show only feature vs state correlations
            annot=True, fmt=".2f", cmap="vlag", center=0,
            cbar_kws={"shrink":.8}
        )
        plt.title(f"Patient {self.patient_id} — Feature vs. Cluster Correlations")
        plt.xlabel("Cluster (one‑hot)")
        plt.ylabel("Features")
        plt.tight_layout()
        plt.show()




sns.set_theme(style="whitegrid", font_scale=1.1)
class DistributionMSM_Strides:
    """
    MSM for gait stability from single-stride distributions
    """
    def __init__(self,
                 patient_id: str,
                 n_states: int = 3,
                 n_bins: int = 64, n_grid_points: int = 100):
        self.patient_id        = patient_id
        self.n_states          = n_states
        self.n_bins            = n_bins
        self.n_grid_points     = n_grid_points
        self.scaler            = StandardScaler()
        self.gmm               = None
        self.transition_matrix = None
        self.stationary_dist   = None
        self.state_labels      = {0:"Stable",1:"Marginally Stable",2:"Unstable"}
        self.metrics_history   = None
        self.validation_metrics = {}  # Store validation metrics
        self.histogram_data    = None  # Store original histogram data

    def compute_stride_metrics(self,
                               signal: np.ndarray,
                               events: np.ndarray,
                               fs: float) -> pd.DataFrame:
        """
        Build one histogram per stride, from one event to the next.
        """
        # edges = np.linspace(signal.min(), signal.max(), self.n_bins+1)
        edges = np.linspace(signal.min(), signal.max(), self.n_bins + 1)
        common_grid = np.linspace(0, 1, self.n_grid_points)
        recs = []
        histograms = []
        
        for i in range(len(events)-1):
            start_idx = int(events[i] * fs)
            end_idx   = int(events[i+1] * fs)
            # skip if out of bounds
            if start_idx<0 or end_idx>len(signal):
                continue
            stride = signal[start_idx:end_idx]
            # skip too-short/long strides
            dur = (events[i+1] - events[i])
            if not (0.3 < dur < 2.0):
                continue
            
            # —— time‐normalize to [0,1] on a fixed grid:
            orig_grid = np.linspace(0, 1, stride.shape[0])
            f_interp  = interp1d(orig_grid, stride,
                                kind='cubic',
                                 fill_value='extrapolate')
            stride = f_interp(common_grid)

            h, _ = np.histogram(stride, bins=edges, density=True)
            h    = h / h.sum()  # Normalize to probability
            histograms.append(h)
            
            rec = {
                'stride_idx': i,
                'time': events[i],        # start time of stride
                'duration': dur
            }
            # add bin probabilities
            for b in range(self.n_bins):
                rec[f'bin_{b}'] = h[b]
            recs.append(rec)

        self.metrics_history = pd.DataFrame(recs)
        self.histogram_data = np.array(histograms)  # Store for validation
        
        print(f"Created histograms from {len(recs)} valid strides")
        print(f"Average stride duration: {self.metrics_history['duration'].mean():.3f}s ± {self.metrics_history['duration'].std():.3f}s")
        
        return self.metrics_history

    def _compute_gmm_reconstruction_error(self):
        """
        Compute reconstruction error from GMM clustering.
        For each stride, reconstruct its histogram from the assigned cluster's mean.
        """
        try:
            if self.gmm is None or self.histogram_data is None:
                print("Warning: GMM or histogram data not available")
                self.validation_metrics['gmm_reconstruction_rmse'] = np.nan
                return
            
            # Get the standardized data that was used for clustering
            cols = [c for c in self.metrics_history.columns if c.startswith('bin_')]
            X_scaled = self.scaler.transform(self.metrics_history[cols].values)
            
            # Get cluster assignments
            cluster_labels = self.gmm.predict(X_scaled)
            
            # Reconstruct each histogram using its cluster mean (in scaled space)
            cluster_means = self.gmm.means_
            reconstructed_scaled = cluster_means[cluster_labels]
            
            # Transform back to original space
            reconstructed_original = self.scaler.inverse_transform(reconstructed_scaled)
            
            # Ensure reconstructed histograms are valid probabilities (non-negative, sum to 1)
            reconstructed_original = np.maximum(0, reconstructed_original)
            row_sums = reconstructed_original.sum(axis=1, keepdims=True)
            row_sums[row_sums == 0] = 1  # Avoid division by zero
            reconstructed_original = reconstructed_original / row_sums
            
            # Compare with original histograms
            original_histograms = self.metrics_history[cols].values
            
            # Compute RMSE
            mse = np.mean((original_histograms - reconstructed_original)**2)
            rmse = np.sqrt(mse)
            
            # Compute per-stride RMSE
            per_stride_mse = np.mean((original_histograms - reconstructed_original)**2, axis=1)
            per_stride_rmse = np.sqrt(per_stride_mse)
            
            self.validation_metrics['gmm_reconstruction_rmse'] = rmse
            self.validation_metrics['gmm_reconstruction_rmse_per_stride'] = per_stride_rmse
            
            print(f"GMM reconstruction RMSE: {rmse:.4f}")
            
        except Exception as e:
            print(f"Warning: Could not compute GMM reconstruction error: {e}")
            self.validation_metrics['gmm_reconstruction_rmse'] = np.nan

    def _compute_clustering_silhouette_score(self, X: np.ndarray, labels: np.ndarray):
        """Compute silhouette score for clustering validation."""
        try:
            # Compute silhouette score
            sil_score = silhouette_score(X, labels)
            self.validation_metrics['clustering_silhouette_score'] = sil_score
            
            print(f"Clustering silhouette score: {sil_score:.3f}")
            
            # Interpretation
            if sil_score > 0.7:
                interpretation = "Strong clustering"
            elif sil_score > 0.5:
                interpretation = "Reasonable clustering"
            elif sil_score > 0.25:
                interpretation = "Weak clustering"
            else:
                interpretation = "Poor clustering"
            
            print(f"Clustering quality: {interpretation}")
            
        except Exception as e:
            print(f"Warning: Could not compute silhouette score: {e}")
            self.validation_metrics['clustering_silhouette_score'] = np.nan


    def _compute_histogram_variance_captured(self):
        """ Compute variance explained by clustering."""


        cols = [c for c in self.metrics_history.columns if c.startswith('bin_')]
        data = self.metrics_history[cols].values  # (N, B)
        labels = self.metrics_history['state'].values

        # 1) Global mean
        global_mean = data.mean(axis=0)

        # 2) TSS
        TSS = np.sum((data - global_mean)**2)

        # 3) WSS
        WSS = 0.0
        for k in range(self.n_states):
            mask = labels == k
            if mask.sum() == 0:
                continue
            cluster_data = data[mask]
            cluster_mean = cluster_data.mean(axis=0)
            WSS += np.sum((cluster_data - cluster_mean)**2)

        # 4) R^2
        R2 = 1 - WSS / TSS if TSS > 0 else 0.0
        self.validation_metrics['variance_explained_by_clustering'] = R2
        print(f"Variance explained by clustering: {R2:.1%}")


    def identify_stability_states(self) -> np.ndarray:
        """ Identify stability states using GMM clustering on stride histograms.
        """
        cols = [c for c in self.metrics_history.columns if c.startswith('bin_')]
        X    = self.scaler.fit_transform(self.metrics_history[cols].values)
        self.gmm = GaussianMixture(n_components=self.n_states,
                                   covariance_type='full',
                                   random_state=42)
        raw = self.gmm.fit_predict(X)

        # order clusters by ascending distance from origin
        norms = np.linalg.norm(self.gmm.means_, axis=1)
        order = np.argsort(norms)
        mapping = {old:new for new,old in enumerate(order)}
        states = np.array([mapping[r] for r in raw])

        self.metrics_history['state'] = states
        
        # Compute validation metrics
        self._compute_gmm_reconstruction_error()
        self._compute_clustering_silhouette_score(X, states)
        self._compute_histogram_variance_captured()
        
        return states

    def compute_transition_matrix(self, states: np.ndarray) -> np.ndarray:
        """ Compute transition matrix from state sequence.
        """
        T = np.zeros((self.n_states, self.n_states))
        for a,b in zip(states[:-1], states[1:]):
            T[a,b] += 1
        row_sums = T.sum(axis=1); row_sums[row_sums==0]=1
        T /= row_sums[:,None]
        self.transition_matrix = T
        return T

    def compute_stationary_distribution(self) -> np.ndarray:
        """ Compute stationary distribution from transition matrix."""
        vals, vecs = eig(self.transition_matrix.T)
        idx        = np.argmin(np.abs(vals-1))
        pi         = np.real(vecs[:,idx])
        pi /= pi.sum()
        self.stationary_dist = np.abs(pi)
        return self.stationary_dist

    def compute_mean_dwell_times(self) -> np.ndarray:
        """ Compute mean dwell times in each state."""
        τ = np.zeros(self.n_states)
        for i in range(self.n_states):
            p = self.transition_matrix[i,i]
            τ[i] = np.inf if p>=1 else 1/(1-p)
        return τ

    def compute_stability_metrics(self) -> dict:
        """ Compute stability metrics."""
        df  = self.metrics_history
        out = {'patient_id':self.patient_id}
        for i in range(self.n_states):
            out[f'frac_state_{i}'] = (df.state==i).mean()
        
        # Add validation metrics
        out.update(self.validation_metrics)
        
        return out

    def visualize_patient_analysis(self):
        """summary: average histograms & transitions with validation metrics."""
        df    = self.metrics_history.copy()
        names = list(self.state_labels.values())
        colors= sns.color_palette(n_colors=self.n_states)

        fig, axes = plt.subplots(2,2, figsize=(12,10))
        
        # 1) Average histogram
        for st in range(self.n_states):
            sub = df[df.state==st]
            cols= [c for c in sub.columns if c.startswith('bin_')]
            mean_hist = sub[cols].mean().values
            centers   = np.linspace(0,1,self.n_bins)
            axes[0,0].plot(centers, mean_hist, label=names[st], color=colors[st])
        axes[0,0].set_title("Average Distribution by State")
        axes[0,0].legend()
        
        # Add validation metrics as text
        if 'gmm_reconstruction_rmse' in self.validation_metrics:
            rmse = self.validation_metrics['gmm_reconstruction_rmse']
            if not np.isnan(rmse):
                axes[0,0].text(0.02, 0.98, f'Reconstruction RMSE: {rmse:.4f}', 
                              transform=axes[0,0].transAxes, va='top', 
                              bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

        # 2) Transition heatmap
        sns.heatmap(self.transition_matrix, annot=True, fmt=".2f",
                    xticklabels=names, yticklabels=names,
                    cmap="Blues", ax=axes[0,1])
        axes[0,1].set_title("Transition Probabilities")
        
        # Add silhouette score
        if 'clustering_silhouette_score' in self.validation_metrics:
            sil_score = self.validation_metrics['clustering_silhouette_score']
            if not np.isnan(sil_score):
                axes[0,1].text(0.02, 0.98, f'Silhouette Score: {sil_score:.3f}', 
                              transform=axes[0,1].transAxes, va='top',
                              bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

        # 3) Stationary dist
        axes[1,0].bar(names, self.stationary_dist, color=colors)
        axes[1,0].set_ylim(0,1)
        axes[1,0].set_title("Stationary Distribution")
        for i,v in enumerate(self.stationary_dist):
            axes[1,0].text(i, v+0.02, f"{v:.1%}", ha='center')
        
        # Add variance explained
        if 'variance_explained_by_clustering' in self.validation_metrics:
            var_exp = self.validation_metrics['variance_explained_by_clustering']
            if not np.isnan(var_exp):
                axes[1,0].text(0.02, 0.98, f'Variance Explained: {var_exp:.1%}', 
                              transform=axes[1,0].transAxes, va='top',
                              bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

        # 4) Transition arrows
        from_, to_ = [], []
        for a,b in zip(df.state[:-1], df.state[1:]):
            from_.append(a); to_.append(b)
        counts = pd.DataFrame({'from':from_,'to':to_}).value_counts().reset_index()
        for _, row in counts.iterrows():
            a,b,c = row['from'], row['to'], row[0]
            axes[1,1].arrow(a, b, 0.2,0.2,
                           width=c/len(from_)*0.5,
                           color='navy', alpha=0.7)
        axes[1,1].set_xlim(-0.5,self.n_states-0.5)
        axes[1,1].set_ylim(-0.5,self.n_states-0.5)
        axes[1,1].set_xticks(range(self.n_states))
        axes[1,1].set_yticks(range(self.n_states))
        axes[1,1].set_xticklabels(names)
        axes[1,1].set_yticklabels(names)
        axes[1,1].set_title("State Transition Patterns")

        plt.tight_layout()
        plt.show()

    def fit(self, signal: np.ndarray, events: np.ndarray, fs: float):
        """
        Runs the pipeline:
          1) Histogram per stride
          2) GMM clustering
          3) MSM construction
        """
        print(f"Processing patient {self.patient_id} using stride-based histograms…")
        self.compute_stride_metrics(signal, events, fs)
        print("Identifying stability states via GMM…")
        states = self.identify_stability_states()
        print("Computing transition matrix…")
        self.compute_transition_matrix(states)
        print("Computing stationary distribution…")
        self.compute_stationary_distribution()
        metrics = self.compute_stability_metrics()
        
        print(f"\nStability metrics for {self.patient_id}:")
        for key, value in metrics.items():
            if key != 'patient_id':
                if isinstance(value, (int, float)) and not np.isnan(value):
                    print(f"  {key}: {value:.3f}")
        
        # Print validation summary
        print(f"\nValidation Summary:")
        if 'gmm_reconstruction_rmse' in self.validation_metrics:
            rmse = self.validation_metrics['gmm_reconstruction_rmse']
            if not np.isnan(rmse):
                print(f"  GMM reconstruction quality: RMSE = {rmse:.4f}")
        if 'clustering_silhouette_score' in self.validation_metrics:
            sil = self.validation_metrics['clustering_silhouette_score']
            if not np.isnan(sil):
                print(f"  Clustering quality: Silhouette = {sil:.3f}")
        if 'variance_explained_by_clustering' in self.validation_metrics:
            var_exp = self.validation_metrics['variance_explained_by_clustering']
            if not np.isnan(var_exp):
                print(f"  Variance explained by clustering: {var_exp:.1%}")
        
        return self
    

    def plot_gmm_reconstruction_error_distribution(self, bins=20):
        """ Plot distribution of GMM reconstruction RMSE per stride."""
        errs = self.validation_metrics.get('gmm_reconstruction_rmse_per_stride')
        if errs is None:
            print("No per‑stride GMM RMSE data.")
            return
        plt.figure(figsize=(8,4))
        sns.histplot(errs, bins=bins, kde=True)
        plt.title("Per‑Stride GMM Reconstruction RMSE")
        plt.xlabel("RMSE")
        plt.ylabel("Count")
        plt.tight_layout()
        plt.show()
            